# TurnWave Phase 5 — retrain on conversational audio

Phase 4's model scored **AP 0.945** on its own test set and **AUC 0.563** on
LiveKit's eot-bench — near-random on real conversation. The cause: its training
corpus was built from a text-to-speech/ASR alignment dataset, so it learned
*"has this sentence finished being read aloud"* rather than *"has this person
finished their turn."*

This phase changes **only the training data** — to `pipecat-ai/smart-turn-data-v3.2`,
which carries human-authored end-of-turn labels from real voice-agent contexts.
Same window, same model, same steps, so the benchmark result is attributable.

**Built to survive session death.** Checkpoints live on Drive and training
resumes automatically: if a session dies, re-run the cells and it picks up where
it stopped. Runs on Colab or Kaggle (30 GPU-hours/week free).

In [ ]:
# 1. Environment, persistent storage, and dependencies.
import os, subprocess, sys, shutil

def run(cmd, what):
    """Fail at the point of failure. A quiet pip error here reappears ten
    minutes later as something baffling and unrelated."""
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:])
        raise SystemExit(f'{what} failed (exit {r.returncode})')
    print(f'{what} OK')

ON_KAGGLE = os.path.isdir('/kaggle/working')
if ON_KAGGLE:
    PERSIST, REPO = '/kaggle/working/turnwave-ckpt', '/kaggle/working/turnwave'
else:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST, REPO = '/content/drive/MyDrive/turnwave', '/content/turnwave'
os.makedirs(PERSIST, exist_ok=True)

if not os.path.isdir(REPO):
    run(['git', 'clone', '-q', 'https://github.com/Nikhils-G/turnwave.git', REPO], 'clone')
%cd {REPO}
subprocess.run(['git', 'checkout', '--', '.']); subprocess.run(['git', 'pull', '-q'])

pip = [sys.executable, '-m', 'pip', 'install', '-q']
run(pip + ['--no-deps', '-e', '.'], 'turnwave')
run(pip + ['sentencepiece', 'soundfile', 'datasets', 'onnx', 'onnxruntime', 'onnxscript'], 'deps')

import torch
assert torch.cuda.is_available(), 'No GPU. Colab: Runtime > Change runtime type > T4.'
print(f'GPU {torch.cuda.get_device_name(0)} | checkpoints -> {PERSIST}')

## 2. Feature cache

Streams the corpus rather than downloading all 41 GB. The cache is ~3 GB of
float16 and stays on local disk — Drive is too slow for the random reads training
does — so a dead session costs this step again (~25 min), but never a checkpoint.

In [ ]:
# Skips itself if the cache is already built.
import json, os

CACHE = 'data/audio_smartturn'
if os.path.exists(f'{CACHE}/manifest.json'):
    m = json.load(open(f'{CACHE}/manifest.json'))
    print('cache present:', {s['split']: s['examples'] for s in m['splits']})
else:
    !python scripts/build_audio_dataset.py --source smart-turn --out {CACHE} \
        --max-examples 100000 --max-eval-examples 8000

## 3. Train

`--resume` is passed unconditionally: it continues from `last.pt` when one
exists in Drive and starts fresh otherwise, so **re-running this cell after a
dead session is the entire recovery procedure.** Checkpoints are written every
250 steps, so at most 249 are ever lost.

In [ ]:
AUDIO_OUT = f'{PERSIST}/audio_eot_v2'
!python -m turnwave.train --task audio --cache {CACHE} --out {AUDIO_OUT} \
    --steps 8000 --batch-size 128 --lr 3e-4 --num-workers 2 --resume

## 4. Export, then benchmark

The gate is eot-bench, not the in-domain score — that is the mistake Phase 4
made. Phase 4 tied the VAD baseline at 55.6% / 21.7% with AUC 0.563; success is
a cutoff rate clearly below VAD.

In [ ]:
!python -m turnwave.export --ckpt {AUDIO_OUT}/best.pt --out-dir {PERSIST}/onnx
!python scripts/plot_training.py {AUDIO_OUT}/log.csv --out {PERSIST}/audio_v2_curves.png
from IPython.display import Image, display
display(Image(f'{PERSIST}/audio_v2_curves.png'))

In [ ]:
# eot-bench pins numpy<2, so it is installed last and the runtime restarted
# after. Installing it earlier lets a later resolve pull numpy 2.x back in.
run([sys.executable, '-m', 'pip', 'install', '-q',
     'git+https://github.com/livekit/eot-bench', 'numpy<2'], 'eot-bench')
print('Now: Runtime > Restart session, then run the LAST cell only.')

In [ ]:
# Run this after the restart.
import os, numpy, shutil
ON_KAGGLE = os.path.isdir('/kaggle/working')
PERSIST = '/kaggle/working/turnwave-ckpt' if ON_KAGGLE else '/content/drive/MyDrive/turnwave'
REPO = '/kaggle/working/turnwave' if ON_KAGGLE else '/content/turnwave'
%cd {REPO}
assert numpy.__version__.startswith('1.'), f'numpy {numpy.__version__}: restart the runtime'
assert shutil.which('eot-harness'), 'harness missing: re-run the install cell'

os.makedirs('checkpoints/onnx', exist_ok=True)
os.makedirs('checkpoints/tokenizer', exist_ok=True)
shutil.copy(f'{PERSIST}/onnx/audio_eot.onnx', 'checkpoints/onnx/audio_eot.onnx')
BASE = 'https://github.com/Nikhils-G/turnwave/releases/download/models-v1'
!wget -q {BASE}/spm.model -O checkpoints/tokenizer/spm.model

os.environ['TURNWAVE_ONNX'] = 'checkpoints/onnx/audio_eot.onnx'
!eot-harness predict --path livekit/eot-bench-data --name all --split validation \
    --adapter turnwave.eot_bench_adapter:TurnWaveAudioOnlyAdapter --output-dir output

import glob
for p in glob.glob('output/**/predictions.parquet', recursive=True):
    !eot-harness compute-metrics --predictions "{p}" --output-dir "{os.path.dirname(p)}/metrics"
ROOT = 'output/livekit__eot-bench-data__validation__min_silence_100ms/en'
!eot-harness compare-models {ROOT}
!cp -r output {PERSIST}/eot_bench_v2